# Module 5 — SOLUTION: Advanced RAG & Vibe Coding

> **Instructor note:** Complete solution for all exercises in Module 5.

In [ ]:
import json
import os

from openai import OpenAI
import chromadb
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder

load_dotenv("../.env")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag", metadata={"hnsw:space": "cosine"}
)
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.com/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")

def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q_emb = embed_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[q_emb], n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {"text": t, "source": m["source"], "similarity": 1 - d}
        for t, m, d in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]

print(f"Collection: {collection.count()} docs")

In [ ]:
# ── Solution 5.1: Query Expansion ─────────────────────────────────────────
def expand_query(question: str, n: int = 3) -> list[str]:
    prompt = f"""Generate {n} alternative phrasings of this question for semantic search.
Return ONLY a JSON array of strings, no explanation.

Question: {question}"""

    response = llm_client.chat.completions.create(
        model=FAST_MODEL, max_tokens=256,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip()
    # Strip markdown code block if present
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]

    try:
        alternatives = json.loads(text)
        return [question] + [a for a in alternatives[:n] if isinstance(a, str)]
    except json.JSONDecodeError:
        return [question]


def retrieve_multi_query(question: str, top_k_per_query: int = 3) -> list[dict]:
    queries = expand_query(question, n=2)
    seen: dict[str, dict] = {}

    for q in queries:
        for chunk in retrieve(q, top_k=top_k_per_query):
            key = chunk["text"]
            if key not in seen or chunk["similarity"] > seen[key]["similarity"]:
                seen[key] = chunk

    return sorted(seen.values(), key=lambda x: x["similarity"], reverse=True)


q = "What does DKV hospitalisation insurance cover?"
print("Expanded queries:", expand_query(q))
multi_chunks = retrieve_multi_query(q)
print(f"Multi-query chunks: {len(multi_chunks)} | Sources: {list(set(c['source'] for c in multi_chunks))}")

In [ ]:
# ── Solution 5.2: HyDE ────────────────────────────────────────────────────
def hyde_retrieve(question: str, top_k: int = 5) -> list[dict]:
    hyde_prompt = f"""Write a short factual paragraph (3-5 sentences) that would directly answer this question.
Do not say you are answering a question — just write the paragraph.

Question: {question}"""

    response = llm_client.chat.completions.create(
        model=FAST_MODEL, max_tokens=256,
        messages=[{"role": "user", "content": hyde_prompt}]
    )
    hypothetical_doc = response.choices[0].message.content

    hyde_embedding = embed_model.encode(hypothetical_doc).tolist()
    results = collection.query(
        query_embeddings=[hyde_embedding], n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {"text": t, "source": m["source"], "similarity": 1 - d}
        for t, m, d in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]


test_q = "What physiotherapy sessions does DKV reimburse after hospitalisation?"
baseline_chunks = retrieve(test_q, top_k=5)
hyde_chunks = hyde_retrieve(test_q, top_k=5)

print(f"Baseline sources: {[c['source'] for c in baseline_chunks]}")
print(f"HyDE sources:     {[c['source'] for c in hyde_chunks]}")

In [ ]:
# ── Solution 5.3: Re-ranking ───────────────────────────────────────────────
print("Loading cross-encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded")


def retrieve_with_rerank(question: str, initial_k: int = 20, final_k: int = 5) -> list[dict]:
    candidates = retrieve(question, top_k=initial_k)
    pairs = [(question, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    for chunk, score in zip(candidates, scores):
        chunk["rerank_score"] = float(score)

    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:final_k]


test_q = "What are the reimbursement limits for dental treatments under DKV?"

print("=== Bi-encoder (top 5) ===")
for i, c in enumerate(retrieve(test_q, top_k=5), 1):
    print(f"  [{i}] sim={c['similarity']:.3f} | {c['source']} | {c['text'][:80]}...")

print("\n=== Re-ranked (top 5 from 20) ===")
for i, c in enumerate(retrieve_with_rerank(test_q), 1):
    print(f"  [{i}] rerank={c['rerank_score']:.3f} | {c['source']} | {c['text'][:80]}...")

In [ ]:
# ── Solution 5.4: AdvancedRAGPipeline ─────────────────────────────────────
SYSTEM_PROMPT = """\
You are a knowledgeable insurance and technology assistant.
Answer ONLY using information from the numbered context chunks provided.
Cite sources using the chunk number. If the context is insufficient, say so.
Never fabricate facts."""


def build_prompt(question: str, chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[{i}] Source: {chunk['source']}")
        lines.append(chunk["text"])
        lines.append("")
    return f"Retrieved Context:\n{''.join(lines)}\nQuestion: {question}\n\nAnswer:"


class AdvancedRAGPipeline:
    def __init__(self, collection, embed_model, llm_client, model=FAST_MODEL,
                 top_k=5, use_query_expansion=False, use_hyde=False,
                 use_reranking=False, rerank_initial_k=20):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.use_query_expansion = use_query_expansion
        self.use_hyde = use_hyde
        self.use_reranking = use_reranking
        self.rerank_initial_k = rerank_initial_k
        if use_reranking:
            self.cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def ask(self, question: str) -> dict:
        if self.use_hyde:
            chunks = hyde_retrieve(question, top_k=self.rerank_initial_k if self.use_reranking else self.top_k)
        elif self.use_query_expansion:
            chunks = retrieve_multi_query(question, top_k_per_query=self.top_k)
            if self.use_reranking:
                candidates = chunks[:self.rerank_initial_k]
                pairs = [(question, c["text"]) for c in candidates]
                scores = self.cross_encoder.predict(pairs)
                for c, s in zip(candidates, scores):
                    c["rerank_score"] = float(s)
                chunks = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:self.top_k]
        elif self.use_reranking:
            chunks = retrieve_with_rerank(question, initial_k=self.rerank_initial_k, final_k=self.top_k)
        else:
            chunks = retrieve(question, top_k=self.top_k)

        prompt = build_prompt(question, chunks[:self.top_k])
        response = self.llm_client.chat.completions.create(
            model=self.model,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )
        return {
            "question": question,
            "answer": response.choices[0].message.content,
            "contexts": [c["text"] for c in chunks[:self.top_k]],
            "sources": [c["source"] for c in chunks[:self.top_k]],
        }


# Test
test_q = "What are the waiting periods and exclusions for DKV hospitalisation insurance?"
configs = {
    "baseline": AdvancedRAGPipeline(collection, embed_model, llm_client),
    "+expansion": AdvancedRAGPipeline(collection, embed_model, llm_client, use_query_expansion=True),
    "+reranking": AdvancedRAGPipeline(collection, embed_model, llm_client, use_reranking=True),
}

for name, pipeline in configs.items():
    result = pipeline.ask(test_q)
    print(f"[{name}]")
    print(f"  Sources: {set(result['sources'])}")
    print(f"  {result['answer']}\n")